# 04 · Retrieve — 07 Rate limiting and 429 backoff

**Ported from `clinical-search/services/rcs_rate_limit.py`. `04-llm-chunk-scoring.ipynb` batches 5-40 calls per query and has no limiter -- this is the exact place a free-tier key gets exhausted, and without this the failure looks like a crash rather than a limit.**

Pairs with `nbio.cost_meter()`, which already exists in this repo: the meter
caps *spend* (dollars), this caps *rate* (requests-per-minute and
tokens-per-minute). They catch different failures -- a run can blow through
a rate limit while still being cheap, and vice versa.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `normalize_budget_estimate` | Clamps a token estimate so one request can never exceed an entire TPM window | `normalize_budget_estimate(50_000, tpm=9_000)` |
| `ProviderRateLimiter` | Sliding-window RPM + TPM budget per provider, blocking until a request fits | `limiter.acquire("groq", est_tokens=2_000)` |
| `parse_retry_after_seconds` | Reads a retry delay off a 429 response's headers, or off the error text | `parse_retry_after_seconds(exc)` |
| `sleep_with_backoff` | Waits the server's requested delay, or exponential backoff if none was given | `sleep_with_backoff(attempt=0, retry_after=None)` |


In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / "nbio.py").is_file():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()

## Step 1 — the per-provider limits, without the per-role indirection

Production keys these by `provider:role` (`groq:supervisor` isolated from
`groq:rcs`) because a 70B supervisor call and an 8B scoring call have very
different token budgets and it matters that one role's burst can't starve
another's. This cookbook has no supervisor -- one role, one caller -- so
that indirection is dropped and each provider gets one flat `(rpm, tpm)`
pair. The numbers themselves are unchanged from production.

In [ ]:
DEFAULT_LIMITS: dict[str, tuple[int, int]] = {
    "groq": (28, 11_000),
    "cerebras": (28, 55_000),
    "gemini": (8, 8_000),
    "openai": (50, 80_000),
    "ollama": (120, 500_000),
}

MIN_BUDGET = 200


def get_limits(provider: str) -> tuple[int, int]:
    return DEFAULT_LIMITS.get((provider or "").lower(), (30, 12_000))


print("groq limits (rpm, tpm):", get_limits("groq"))
print("unknown provider falls back to:", get_limits("some-new-provider"))

## Step 2 — `normalize_budget_estimate`: prevent deadlock

Ported verbatim, docstring included -- the reason it exists is the whole
point: without this clamp, a request that legitimately needs more tokens
than an entire TPM window fits would never be let through by
`ProviderRateLimiter.acquire` below. It would wait for a window that can
never have enough room, forever. Clamping the *estimate* (not the real
call) means the limiter proceeds instead of deadlocking; it does not
change what the call actually sends.

In [ ]:
def normalize_budget_estimate(est_tokens: int, tpm: int) -> int:
    """Clamp so one request never exceeds an entire TPM window (prevents deadlock)."""
    est = max(MIN_BUDGET, int(est_tokens))
    cap = max(MIN_BUDGET, int(tpm))
    return cap if est > cap else est


rpm, tpm = get_limits("groq")
print(f"groq tpm window: {tpm}")
print("estimate 2,000 tokens ->", normalize_budget_estimate(2_000, tpm))
print("estimate 50,000 tokens (larger than the whole window) ->", normalize_budget_estimate(50_000, tpm))

## Step 3 — `ProviderRateLimiter`: the sliding window itself

Ported unchanged except for the role-key removal above. One `acquire` call
per real request: if the window has room (both the call count and the
token estimate fit), it's granted immediately; if not, it blocks and
retries until the 60-second window resets. The demo below uses a tiny
override -- `rpm=2` -- so the wait is a few seconds instead of most of a
minute, and times the call to prove it actually blocked rather than just
printing a number.

In [ ]:
import random
import threading
import time


class ProviderRateLimiter:
    """Simple sliding-window RPM + TPM budget per provider."""

    def __init__(self) -> None:
        self._lock = threading.Lock()
        self._window_start = time.monotonic()
        self._calls: dict[str, int] = {}
        self._tokens: dict[str, int] = {}

    def _maybe_reset_window(self, now: float) -> None:
        if now - self._window_start >= 60.0:
            self._window_start = now
            self._calls.clear()
            self._tokens.clear()

    def acquire(self, provider: str, est_tokens: int, *, rpm: int, tpm: int) -> None:
        est_tokens = normalize_budget_estimate(est_tokens, tpm)
        while True:
            with self._lock:
                now = time.monotonic()
                self._maybe_reset_window(now)
                calls = self._calls.get(provider, 0)
                tokens = self._tokens.get(provider, 0)
                if calls + 1 <= rpm and tokens + est_tokens <= tpm:
                    self._calls[provider] = calls + 1
                    self._tokens[provider] = tokens + est_tokens
                    return
                wait = max(0.5, 60.0 - (now - self._window_start) + random.uniform(0.05, 0.35))
            time.sleep(min(wait, 15.0))

## Step 4 — prove it actually blocks, not just accepts

`rpm=2` for this demo only -- the third call in one window must wait for
the window to roll over rather than being granted immediately. Timed with
`time.monotonic()` so the wait is measured, not asserted on faith.

In [ ]:
DEMO_RPM, DEMO_TPM = 2, 5_000  # tiny window so this finishes in seconds, not a minute

limiter = ProviderRateLimiter()
limiter._window_start = time.monotonic() - 55  # start the window near expiry so the demo is short

start = time.monotonic()
for i in range(3):
    call_start = time.monotonic()
    limiter.acquire("demo-provider", est_tokens=500, rpm=DEMO_RPM, tpm=DEMO_TPM)
    elapsed = time.monotonic() - call_start
    print(f"call {i + 1}: granted after {elapsed:.2f}s wait")

total = time.monotonic() - start
print(f"\ntotal: {total:.2f}s for 3 calls at rpm={DEMO_RPM} -- the 3rd call had to wait for a window reset")
assert total > 1.0, "the 3rd call should have measurably waited, not been granted instantly"

## Step 5 — `parse_retry_after_seconds`: read a 429's own retry hint

Ported unchanged. A well-behaved 429 response names its own retry delay,
either in a `Retry-After`-shaped header or in the error text itself
(`"retry after 12 seconds"`). Demonstrated against a small stand-in
exception object shaped like `httpx`'s, so no real network call or actual
429 is needed to prove the parsing works.

In [ ]:
import re


def parse_retry_after_seconds(exc: BaseException) -> float | None:
    """Extract retry delay from OpenAI/httpx 429 responses."""
    resp = getattr(exc, "response", None)
    if resp is not None:
        headers = getattr(resp, "headers", None) or {}
        for key in ("retry-after", "Retry-After", "x-ratelimit-reset-requests"):
            val = headers.get(key)
            if val is None:
                continue
            try:
                return float(val)
            except (TypeError, ValueError):
                m = re.search(r"(\d+(?:\.\d+)?)", str(val))
                if m:
                    return float(m.group(1))
    body = str(exc)
    m = re.search(r"retry(?:\s+after|\s+in)?\s+(\d+(?:\.\d+)?)", body, re.I)
    if m:
        return float(m.group(1))
    return None


class _FakeResponse:
    def __init__(self, headers):
        self.headers = headers


class _FakeHTTPError(Exception):
    def __init__(self, message, headers=None):
        super().__init__(message)
        if headers is not None:
            self.response = _FakeResponse(headers)


with_header = _FakeHTTPError("429 Too Many Requests", headers={"retry-after": "12"})
with_text_only = _FakeHTTPError("rate limited, please retry after 7 seconds")
no_hint = _FakeHTTPError("429 Too Many Requests")

print("from header:    ", parse_retry_after_seconds(with_header))
print("from body text: ", parse_retry_after_seconds(with_text_only))
print("no hint at all: ", parse_retry_after_seconds(no_hint))

assert parse_retry_after_seconds(with_header) == 12.0
assert parse_retry_after_seconds(with_text_only) == 7.0
assert parse_retry_after_seconds(no_hint) is None

## Step 6 — `sleep_with_backoff`: honor the hint, or fall back to exponential

Ported unchanged. If the server gave a retry delay, use it (plus a little
jitter, so many clients retrying at once don't all wake up in the same
instant). If it didn't, back off exponentially by attempt number. The demo
below checks the *computed delay*, not a real multi-second sleep, for the
attempt=3 case -- only attempt=0's small real delay is actually slept.

In [ ]:
def compute_backoff_delay(attempt: int, retry_after: float | None) -> float:
    """The delay `sleep_with_backoff` would use -- split out from the real
    sleep so this can be checked without waiting through it."""
    if retry_after is not None and retry_after > 0:
        return retry_after + random.uniform(0.05, 0.25)
    return min(30.0, (2 ** attempt) + random.uniform(0, 0.5))


def sleep_with_backoff(attempt: int, retry_after: float | None) -> None:
    time.sleep(compute_backoff_delay(attempt, retry_after))


delay_with_hint = compute_backoff_delay(attempt=3, retry_after=5.0)
delay_no_hint_attempt0 = compute_backoff_delay(attempt=0, retry_after=None)
delay_no_hint_attempt4 = compute_backoff_delay(attempt=4, retry_after=None)

print(f"server said retry in 5s (attempt doesn't matter): ~{delay_with_hint:.2f}s")
print(f"no hint, attempt 0 (2**0=1): ~{delay_no_hint_attempt0:.2f}s")
print(f"no hint, attempt 4 (2**4=16): ~{delay_no_hint_attempt4:.2f}s")

print("\nactually sleeping the small attempt-0 case to prove sleep_with_backoff runs:")
t0 = time.monotonic()
sleep_with_backoff(attempt=0, retry_after=None)
print(f"slept {time.monotonic() - t0:.2f}s")

## Where this runs in the pipeline

`04-llm-chunk-scoring.ipynb` is the highest-volume model call in this
stage -- 5 to 40 scoring calls for a single query. Wrapping its live-scoring
loop in `limiter.acquire(provider, est_tokens, rpm=..., tpm=...)` before
each call is the integration point; this notebook only proves the limiter
itself works, correctly, in isolation.